[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/gravitational_lensing/gravitational_lensing.ipynb)

# Gravitational Lensing by Point Masses

This example computes the multiple sky images, critical curves, and caustics produced when light from a background source passes a pair of point masses. As the source moves behind the binary lens, its images stretch into arcs, pairs of images appear or merge whenever the source crosses the caustic, and total brightness rises and falls, following M. Dominik, [*The binary gravitational lens and its extreme cases*](https://arxiv.org/abs/astro-ph/9903014) (1999), section 2.

Directions on the sky are modelled as intrinsic points (`Point-`) in the Euclidean plane in units of the total mass's Einstein angle, and each mass deflects a sightline towards itself by its mass times the inverse separation, `separation.inverse()`. Differentiating an inverse sandwiches a small step between two inverses with a minus sign, so the sky-to-source Jacobian is the identity plus one scaled separation-line reflection per mass, `Point + (masses * (inverse_separation * Point * inverse_separation)).sum(axis=-1)`; planar reflections are traceless, giving trace two everywhere, while `local.det()` gives the signed area ratio that vanishes on the critical curve.

In complex lensing notation, sky and source directions are packed as $z = \theta_1 + i\theta_2$ and $\zeta = \beta_1 + i\beta_2$ with conjugated deflection $\zeta = z - \sum_k m_k / (\bar z - \bar z_k)$, while in real $2 \times 2$ Jacobian coordinates $\partial\boldsymbol\beta / \partial\boldsymbol\theta = I - \sum_k m_k |\boldsymbol\theta - \boldsymbol\ell_k|^{-4} \bigl(|\boldsymbol\theta - \boldsymbol\ell_k|^2 I - 2(\boldsymbol\theta - \boldsymbol\ell_k)(\boldsymbol\theta - \boldsymbol\ell_k)^{\mathsf T}\bigr)$ separates the convergence and shear components to evaluate $\det(\partial\boldsymbol\beta / \partial\boldsymbol\theta) = 1 - |\gamma|^2$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Callable, Iterator

import numpy as np

from numga import NumpyContext
from numga.algebras import VGA2D as ga
from examples.relativity.gravitational_lensing import render, scenarios
from examples.util.animation import save_figure, show_states

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                    # Scalar+
# Sky directions are intrinsic planar points (`Antivector-`).
Point = ga.gatype.antivector()                                                 # Antivector-
LocalMap = ga.gatype((Point, Point))                                           # Point- <- Point-
Exposure = tuple[Callable[[Point], Scalar], Callable[[Point], Scalar]]

## Sightlines traced back to the source

Each point mass bends a passing ray towards itself by an angle inversely proportional to its separation on the sky. Because `separation.inverse()` points along `separation` with length `1 / |separation|`, subtracting the mass-weighted inverse separations from an observed sky direction gives the source direction reached by that ray.

In [ ]:
n_masses = scenarios.n_masses
positions = scenarios.positions                                                # [n_masses] Point-
masses = scenarios.masses                                                      # [n_masses]
sky_half_width = scenarios.sky_half_width
n_pixels = scenarios.n_pixels

In [ ]:
def sky_grid(half_width: float, n_pixels: int) -> Point:
    """Pixel centres across `[-half_width, half_width]` form a square grid of intrinsic sky directions (`[n_pixels, n_pixels] Point-`)."""
    angles = (np.arange(n_pixels) + 0.5) * (2 * half_width / n_pixels) - half_width
    return (mv.x * angles[None, :] + mv.y * angles[:, None]).dual()            # [n_pixels, n_pixels] Point-


def source_direction(observed: Point, positions: Point, masses: np.ndarray) -> Point:
    """Each mass in `positions` (`[n_masses] Point-`) bends `observed` (`[...] Point-`) by its mass times the inverse separation to reach the source direction (`[...] Point-`)."""
    separation = observed[..., None] - positions                               # [..., ...] Point-
    return observed - (masses * separation.inverse()).sum(axis=-1)             # [...] Point-


directions = sky_grid(sky_half_width, n_pixels)                                # [n_pixels, n_pixels] Point-

## Local derivative map and critical curve

Perturbing a sky direction by a small displacement perturbs each separation by the same step, and the first-order change of an inverse is minus the step sandwiched between two inverses, `-(inverse_separation * step * inverse_separation)`. Each mass therefore contributes a reflection across its separation line scaled by the inverse squared distance, leaving `local.trace() == 2` everywhere while `local.det()` measures the signed sky-to-source area ratio that vanishes along the critical curve.

In [ ]:
def local_map(observed: Point, positions: Point, masses: np.ndarray) -> LocalMap:
    """The sky-to-source Jacobian (`[...] Point- <- Point-`) sums the identity with one scaled separation-line reflection per mass."""
    inverse_separation = (observed[..., None] - positions).inverse()           # [..., ...] Point-
    # Differentiating separation.inverse() gives minus the step sandwiched between two inverse separations.
    return Point + (masses * (inverse_separation * Point * inverse_separation)).sum(axis=-1)   # [...] Point- <- Point-


local = local_map(directions, positions, masses)                               # [n_pixels, n_pixels] Point- <- Point-
area = local.det()                                                             # [n_pixels, n_pixels] Scalar+

In [ ]:
print("area ratio range:", area.to_array().min(), area.to_array().max())

## Source crossing the six-cusped caustic

Surface brightness is conserved along each ray, so every sky pixel displays the Gaussian source evaluated at the source direction its ray reaches, magnifying total flux wherever images cover a larger solid angle on the sky. Streaming the source along a horizontal chord across the caustic updates both the unlensed source view and the lensed sky view in one pass.

In [ ]:
n_steps = scenarios.n_steps
path = scenarios.path                                                          # [n_steps] Point-
shown_step = scenarios.shown_step
source_width = scenarios.source_width
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
def source_brightness(directions: Point, centre: Point, width: float) -> Scalar:
    """A round Gaussian source centred at `centre` with angular standard deviation `width` has unit peak brightness (`[...] Scalar+`)."""
    offset = directions - centre                                               # [...] Point-
    return (-(offset | offset) / (2 * width**2)).exp()                         # [...] Scalar+


def path_exposures(path: Point, positions: Point, masses: np.ndarray, width: float) -> Iterator[Exposure]:
    """Yields `(unlensed, lensed)` sky brightness maps along each source centre in `path` (`[n_steps] Point-`)."""
    for centre in path:
        yield (
            lambda directions, centre=centre: source_brightness(directions, centre, width),
            lambda directions, centre=centre: source_brightness(source_direction(directions, positions, masses), centre, width),
        )


lens = lambda observed: source_direction(observed, positions, masses)
exposures = show_states(
    path_exposures(path, positions, masses, source_width),
    render.lensing_animation,
    directions,
    area,
    lens,
    positions,
    name="gravitational_lensing",
    duration_ms=frame_duration_ms,
    n_frames=n_steps,
);

The left panel shows the moving Gaussian source against the coral six-cusped caustic, and the right panel shows the lensed sky images splitting and merging across the cyan critical curve around the two grey lens masses.

In [ ]:
save_figure(render.lensing_figure(directions, area, lens, positions, exposures[shown_step]), "gravitational_lensing");

At `shown_step`, the source straddles a fold of the coral caustic on the left while two bright images merge into an elongated arc across the cyan critical curve on the right.

In [ ]:
# checks
assert ((local.trace() - 2.0).abs() < 1e-11).all()
probe = mv.x.dual()                                                            # Point-
recovered = ((2 * Point - local) / area)(local(probe))                         # [n_pixels, n_pixels] Point-
away_from_critical = area.abs() > 1e-2
assert (((recovered - probe).scalar_norm_squared() < 1e-18) | ~away_from_critical).all()
assert len(exposures) == n_steps